# 01 - Quickstart: rewrite a passage end to end
First run downloads the 7B infiller (~15 GB) and quantizes it to 4-bit: expect several minutes. For a fast first check, set `CONFIG = "configs/smoke.yaml"` (lower quality).

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])

In [ ]:
import os
REPO = "https://github.com/deymonp/sira-lab.git"
if not os.path.exists("/content/sira-lab"):
    !git clone -q {REPO} /content/sira-lab
else:
    !git -C /content/sira-lab pull -q
%cd /content/sira-lab
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

In [ ]:
from sira_lab.config import load_config
from sira_lab.factory import build_rewriter

CONFIG = "configs/default.yaml"      # or "configs/smoke.yaml" for tiny models
cfg = load_config(CONFIG)
rewriter = build_rewriter(cfg)
import torch; print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

## Run it on a sample paragraph (replace with your own text, up to ~1000 words)

In [ ]:
SAMPLE = """Evaluating the robustness of statistical text watermarks requires a controlled comparison between watermarked generations and their rewritten counterparts. In this study we generate passages with a keyed sampling scheme, apply several paraphrasing strategies of increasing strength, and measure how much detection power remains. Our preliminary analysis suggests that the positions carrying the most information about the key are also the positions where a language model is least certain about the next word. Consequently, targeted rewriting of those positions may disrupt the signal while leaving most of the sentence untouched. We report detection rates at a fixed false positive rate and quantify semantic drift using sentence embeddings."""

import time
from IPython.display import Markdown, display
t = time.time()
rep = rewriter.rewrite(SAMPLE)
display(Markdown(rep.to_markdown()))
print(rep.text_out)

## What changed?

In [ ]:
for c in rep.chunks:
    print(f"[{c.index}] {c.status}  attempts={c.attempts}  sim={c.sim}  masked={c.n_masked}")
    print("   swaps:", c.swaps[:15])
    for w in c.warnings: print("   warn:", w)

## Save the run so we can compare configurations later

In [ ]:
import json
json.dump(rep.to_dict(), open("/content/run_001.json", "w"), indent=2)
print("saved /content/run_001.json  (download it from the file browser, or send me the printed summary above)")

## Optional: launch the web UI with a temporary public link (expires; for colleagues' quick tests only)

In [ ]:
from sira_lab.ui import build_ui
demo = build_ui(rewriter)
demo.queue().launch(share=True, debug=False)